# 1. Importando bilbliotecas

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
from sklearn.metrics import ConfusionMatrixDisplay
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier

# 2. Carregando o conjunto de dados

In [ ]:
df = pd.read_csv("/content/WineQT.csv")

## Visão geral dos dados

Examinar estrutura do dataset, tipos de dados e presença ou não de valores nulos e duplicados.

In [ ]:
print("Dimensões:", df.shape)

df.info()

df.describe().T

### Descrição do Dataset

O Wine Quality Dataset contém informações físico-químicas de vinhos tintos e sua respectiva avaliação de qualidade.

O conjunto possui:

- 1.599 amostras
- 11 variáveis preditoras
- 1 variável alvo (quality)

As variáveis representam propriedades como:

- Fixed Acidity
- Volatile Acidity
- Citric Acid
- Residual Sugar
- Chlorides
- Free Sulfur Dioxide
- Total Sulfur Dioxide
- Density
- pH
- Sulphates
- Alcohol

A variável alvo (`quality`) representa uma nota entre 0 e 10 atribuída ao vinho.

In [ ]:
# excluindo a coluna de Id pois é irrelevante para a análise
df.drop(columns=["Id"], inplace=True)

In [ ]:
plt.figure(figsize=(8,5))
sns.countplot(x="quality", data=df)
plt.title("Distribuição da qualidade dos vinhos")
plt.xlabel("Qualidade")
plt.ylabel("Quantidade")
plt.show()



*   A maioria dos vinhos possui notas 5 e 6.
*   Existem poucos vinhos de qualidade muito alta ou muito baixa.
*   O conjunto apresenta desbalanceamento entre as classes.







# 3. Análise exploratória de dados

A maioria das variáveis apresenta distribuição assimétrica, indicando concentração de observações em determinados intervalos.

In [ ]:
df.hist(figsize=(15,10), bins=20)
plt.tight_layout()
plt.show()

## Demonstrando as correlações entre as variáveis

In [ ]:
plt.figure(figsize=(15, 6))
corr = sns.heatmap(df.corr(), annot=True, cmap="Spectral_r")
plt.title("Correlação entre as variáveis")
plt.show()


*   Álcool possui correlação positiva com a qualidade;
*   Acidez volátil possui correlação negativa;
*   Algumas variáveis apresentam alta correlação entre si.






## Identificando outliers

In [ ]:
import math

colunas = df.select_dtypes(include="number").columns
n = len(colunas)

ncols = 3
nrows = math.ceil(n / ncols)

fig, axes = plt.subplots(nrows, ncols, figsize=(15, 4 * nrows))
axes = axes.flatten()

for i, coluna in enumerate(colunas):
    axes[i].boxplot(df[coluna].dropna())
    axes[i].set_title(coluna)

for j in range(i + 1, len(axes)):
    fig.delaxes(axes[j])

plt.tight_layout()
plt.show()

In [ ]:
for coluna in df.select_dtypes(include="number").columns:
    Q1 = df[coluna].quantile(0.25)
    Q3 = df[coluna].quantile(0.75)
    IQR = Q3 - Q1

    inferior = Q1 - 1.5 * IQR
    superior = Q3 + 1.5 * IQR

    quantidade = df[
        (df[coluna] < inferior) |
        (df[coluna] > superior)
    ].shape[0]

    print(f"{coluna}: {quantidade} outliers")

In [ ]:
plt.figure(figsize=(8,5))

sns.boxplot(
    x="quality",
    y="alcohol",
    data=df
)
plt.title("Distribuição do teor alcoólico por qualidade do vinho")
plt.show()

Vinhos com maior qualidade tem maior teor alcoólico

In [ ]:
plt.figure(figsize=(8,5))

sns.boxplot(
    x="quality",
    y="volatile acidity",
    data=df
)
plt.title("Distribuição da volatilidade ácida por qualidade do vinho")

plt.show()

Quanto maior a qualidade do vinho maior a volatilidade ácida.

## Transformação da variável qualidade em variável binária:


Atribuimos o valor 1 para observações com quality >= 7 e 0 caso contrário.



In [ ]:
df["quality_binary"] = df["quality"].apply(
    lambda x: 1 if x >= 7 else 0
)

In [ ]:
sns.countplot(
    x="quality_binary",
    data=df
)
plt.title("Distribuição da qualidade binária dos vinhos")
plt.show()

In [ ]:
grouped = df.groupby("quality_binary").mean()
grouped

In [ ]:
df.groupby("quality_binary").describe()

In [ ]:
X = df.drop(columns=["quality", "quality_binary"])
y = df["quality_binary"]

# 3. Pré-processamento de dados


In [ ]:
df.isnull().sum()

### Divisão dos dados em conjunto de teste e treino

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

Padronização dos dados para melhorar o desempenho de alguns algoritmos

In [ ]:
scaler = StandardScaler()
scaler.fit(X_train)
X_train_standard_scaled = scaler.transform(X_train)
X_test_standard_scaled = scaler.transform(X_test)

# 4. Treinamento dos modelos

### 4.1 KNN

In [ ]:
modelo_classificador = KNeighborsClassifier(n_neighbors=3)
modelo_classificador.fit(X_train_standard_scaled, y_train)

In [ ]:
y_pred = modelo_classificador.predict(X_test_standard_scaled)

In [ ]:
ConfusionMatrixDisplay.from_predictions(
    y_test,
    y_pred,
    cmap="Blues"
)

plt.title("Matriz de Confusão - KNN")
plt.show()

### 4.2 Random Forest

In [ ]:
modelo_rf = RandomForestClassifier(random_state=42)

modelo_rf.fit(X_train, y_train)

y_pred_rf = modelo_rf.predict(X_test)

In [ ]:
ConfusionMatrixDisplay.from_predictions(
    y_test,
    y_pred_rf,
    cmap="Greens"
)

plt.title("Matriz de Confusão - Random Forest")
plt.show()

A matriz de confusão mostra que o modelo Random Forest apresentou um bom desempenho na classificação dos vinhos. Dos 197 vinhos de baixa/média qualidade presentes no conjunto de teste, 192 foram classificados corretamente, enquanto apenas 5 foram classificados incorretamente como de boa qualidade. Em relação aos vinhos de boa qualidade, o modelo identificou corretamente 19 dos 32 casos, errando 13 classificações.
Apesar da elevada acurácia (92%), a matriz de confusão revela que o modelo apresentou melhor desempenho na identificação de vinhos de baixa/média qualidade do que de vinhos de boa qualidade. Esse comportamento pode ser explicado pelo desbalanceamento das classes no conjunto de dados, já que havia um número significativamente maior de exemplos da classe 0 durante o treinamento.

# 5. Avaliação dos modelos

In [ ]:
print(f"Resultados para KNN:")
print("Acurácia:", accuracy_score(y_test, y_pred))
print("Matriz de Confusão:\n", confusion_matrix(y_test, y_pred))
print("Relatório de Classificação:\n", classification_report(y_test, y_pred))

In [ ]:
print(f"Resultados para Random Forest:")
print("Acurácia:", accuracy_score(y_test, y_pred_rf))
print("Matriz de Confusão:\n", confusion_matrix(y_test, y_pred_rf))
print("Relatório de Classificação:\n", classification_report(y_test, y_pred_rf))

Após a avaliação dos modelos, foi realizada uma predição utilizando o Random Forest, por ter apresentado o melhor desempenho. A partir das características físico-químicas informadas, o modelo classificou o vinho como de boa qualidade (ou baixa/média qualidade).

In [ ]:
novo_vinho = X.iloc[[5]]

In [ ]:
previsao = modelo_rf.predict(novo_vinho)

print(previsao)

In [ ]:
if previsao[0] == 1:
    print("O vinho é de boa qualidade.")
else:
    print("O vinho é de baixa/média qualidade.")

# 6. Resultados


### 6.1 Identificando a importância das variáveis

In [ ]:
importancias = pd.Series(
    modelo_rf.feature_importances_,
    index=X.columns
)

importancias = importancias.sort_values()

plt.figure(figsize=(8,6))
importancias.plot(kind="barh")
plt.title("Importância das Variáveis - Random Forest")
plt.xlabel("Importância")
plt.show()

Após o treinamento do modelo Random Forest, foi realizada a análise da importância das variáveis (feature importance), com o objetivo de identificar quais características físico-químicas tiveram maior influência na classificação dos vinhos. Os resultados indicam que o teor alcoólico (Alcohol) foi a variável mais relevante para as decisões do modelo. Em seguida, destacam-se Sulphates, Volatile Acidity e Citric Acid, que também contribuíram significativamente para a classificação da qualidade dos vinhos.
Esse resultado está alinhado com a análise exploratória realizada anteriormente. Durante a EDA, observou-se que a variável Alcohol apresentava correlação positiva com a qualidade dos vinhos, enquanto Volatile Acidity apresentava correlação negativa. A análise de importância das variáveis reforça que essas características exercem influência significativa no desempenho do modelo.
A análise de importância das variáveis permitiu compreender quais características físico-químicas foram mais relevantes para o modelo Random Forest. Além de apresentar a maior acurácia entre os algoritmos avaliados (92%), o Random Forest também oferece maior interpretabilidade, indicando que o teor alcoólico, os sulfatos e a acidez volátil foram os principais fatores considerados na classificação dos vinhos.